<a href="https://colab.research.google.com/github/EceNurKizikli/Nyc-Airbnb-Price-Prediction/blob/main/NYC_Airbnb_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA CLEANING



**1.   Take the data**



Kaggle API

In [ ]:
import os

# Paste your token inside the quotes below
os.environ['KAGGLE_API_TOKEN'] =  "YOUR_KAGGLE_TOKEN_HERE"

# Now you can run the download command
!kaggle datasets download -d dgomonov/new-york-city-airbnb-open-data

# Unzip the data
!unzip new-york-city-airbnb-open-data.zip -d nyc_data

Read the csv

In [ ]:
import pandas as pd
df = pd.read_csv('nyc_data/AB_NYC_2019.csv')
df.head()



**1.    Handling Nulls**




Check where the NaNs are

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

Drop last_review and reviews_per_month since these are heavily missing, and remove the rows that include NaN after that

In [ ]:
# 1. Drop the columns with heavy missing values
df.drop(['last_review', 'reviews_per_month'], axis=1, inplace=True)

# 2. Capture the row count before dropping remaining NaNs
initial_rows = df.shape[0]

# 3. Drop the rows where 'name' or 'host_name' are missing (the ~37 rows)
df.dropna(inplace=True)

# 4. Capture the row count after
final_rows = df.shape[0]

# 5. Calculate the loss
rows_lost = initial_rows - final_rows
percent_lost = (rows_lost / initial_rows) * 100

print(f"Columns dropped: last_review, reviews_per_month")
print(f"Initial row count: {initial_rows}")
print(f"Final row count: {final_rows}")
print(f"Total rows lost: {rows_lost}")
print(f"Percentage of rows lost: {percent_lost:.4f}%")


**2.   Fixing Price Outliers**



Boxplot to see what is an outlier in this data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a boxplot for Price
plt.figure(figsize=(10, 5))
sns.boxplot(x=df['price'])
plt.title('Boxplot of Price to Identify Outliers')
plt.show()

Keep only the bottom 99% of prices

In [ ]:
q_high = df["price"].quantile(0.99)
df_filtered = df[df["price"] < q_high]

print(f"99th percentile is: ${q_high:.2f}")
print(f"New max price is: ${df_filtered['price'].max()}")

In [ ]:
# 1. Save the count before filtering
count_before = len(df)

# 2. Filter the dataframe to keep only prices <= 795
df = df[df['price'] <= 795]

# 3. Calculate how many rows were removed
count_after = len(df)
removed_count = count_before - count_after

print(f"Removed {removed_count} extreme price outliers.")
print(f"New maximum price in the dataset: ${df['price'].max()}")
print(f"Total rows remaining: {count_after}")

Boxplot after removing the outliers

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.boxplot(x=df['price'])
plt.title('Boxplot of Price (After Removing Outliers > $795)')
plt.show()

# DATA PREPROCESSING

Drop Non-Predictive Columns & Encode Categorical Data



In [ ]:
# 1. Select the features mentioned in the problem statement
features = [
    'neighbourhood_group', 'latitude', 'longitude',
    'room_type', 'price', 'minimum_nights',
    'number_of_reviews', 'calculated_host_listings_count', 'availability_365'
]
df_final = df[features].copy()

# 2. One-Hot Encode categorical variables (Boroughs and Room Types)
df_final = pd.get_dummies(df_final, columns=['neighbourhood_group', 'room_type'], drop_first=True)

# 3. View the new columns (e.g., room_type_Private room, neighbourhood_group_Brooklyn)
df_final.head()

Check for Other Outliers

In [ ]:
# Check the top 10 highest values
print("Top 10 highest minimum_nights:")
print(df_final['minimum_nights'].sort_values(ascending=False).head(10))

# Visualizing the distribution
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.boxplot(x=df_final['minimum_nights'])
plt.title('Boxplot of Minimum Nights')
plt.show()

In [ ]:
# 1. Calculate the 99th percentile for minimum_nights
min_nights_99 = df_final['minimum_nights'].quantile(0.99)

# 2. Filter the data
count_before = len(df_final)
df_final = df_final[df_final['minimum_nights'] <= min_nights_99]
count_after = len(df_final)

# 3. Report the results
print(f"99th percentile for minimum_nights is: {min_nights_99} nights.")
print(f"Removed {count_before - count_after} listings that required more than {min_nights_99} nights.")
print(f"New max minimum_nights: {df_final['minimum_nights'].max()}")

# EXPLORATORY DATA ANALYSIS (EDA)

Price Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(15, 5))

# Original Price Distribution
plt.subplot(1, 2, 1)
sns.histplot(df['price'], bins=50, kde=True, color='blue')
plt.title('Original Price Distribution')
plt.xlabel('Price ($)')

# Log-Transformed Price Distribution
plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['price']), bins=50, kde=True, color='green')
plt.title('Log-Transformed Price Distribution')
plt.xlabel('Log(Price + 1)')

plt.tight_layout()
plt.show()

Geographic Influence

In [ ]:
plt.figure(figsize=(10, 8))

scatter = plt.scatter(x=df['longitude'], y=df['latitude'],
                      c=np.log1p(df['price']), cmap='coolwarm', alpha=0.6, s=10)

plt.colorbar(scatter, label='Log(Price)')
plt.title('NYC Airbnb Prices by Coordinates (Log Scale)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

Room Type Impact

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='room_type', y='price', data=df, palette='Set2')
plt.title('Price Distribution by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Price ($)')
plt.show()

Correlation Analysis

In [ ]:
plt.figure(figsize=(12, 10))

corr_matrix = df_final.corr()

sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap of Features')
plt.show()

print(corr_matrix['price'].sort_values(ascending=False))

# Finding Correlated Features
(According to feedback)

In [ ]:
import pandas as pd
import numpy as np

# Calculate the correlation matrix for all features in df_final, excluding 'price'
# This ensures we check multicollinearity among all potential predictor variables.
corr_matrix_features = df_final.drop('price', axis=1).corr().abs()

# Select upper triangle of correlation matrix to avoid duplicates and self-correlations
uppertri_corr = corr_matrix_features.where(np.triu(np.ones(corr_matrix_features.shape), k=1).astype(bool))

# Find features with correlation greater than 0.7
high_corr_features = [column for column in uppertri_corr.columns if any(uppertri_corr[column] > 0.7)]

print("Features with high correlation (absolute value > 0.7) among themselves:")
if high_corr_features:
    for col in high_corr_features:
        # Get the features that are highly correlated with the current 'col'
        highly_correlated_with_col = uppertri_corr.index[uppertri_corr[col] > 0.7].tolist()
        if highly_correlated_with_col:
            for other_col in highly_correlated_with_col:
                correlation_value = corr_matrix_features.loc[col, other_col]
                print(f"  - '{col}' is highly correlated with '{other_col}' (Correlation: {correlation_value:.2f})")
else:
    print("  No highly correlated features found with an absolute correlation greater than 0.7.")

# Display the full correlation matrix for the features
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix_features, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Feature Correlation Matrix (Excluding Price)')
plt.show()

In [ ]:
# Drop one of the highly correlated features to address multicollinearity
# We'll drop 'neighbourhood_group_Brooklyn' as an example

if 'neighbourhood_group_Brooklyn' in df_final.columns:
    df_final = df_final.drop(columns=['neighbourhood_group_Brooklyn'])
    print("Dropped 'neighbourhood_group_Brooklyn' due to high correlation.")
else:
    print("'neighbourhood_group_Brooklyn' not found in DataFrame, no action taken.")

# Display the first few rows of the modified df_final to confirm the change
display(df_final.head())

# DATA SPLITTING

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Define X (Features) and y (Target)
# We drop 'price' from X because the model shouldn't see the answer while learning!
X = df_final.drop('price', axis=1)
y = df_final['price']

# 2. Split into Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, shuffle=True
)

# 3. Split Temp (30%) into Val (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, shuffle=True
)

print("X and y defined and split successfully!")
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit/Transform based on the rules learned from X_train
X_train_scaled = scaler.fit_transform(X_train)

# Use those same rules to scale Val and Test
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Data is split and scaled. You are ready to build the model.")

# LINEAR REGRESSION MODEL

**Train**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Initialize and Train
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make Predictions on the Validation set
y_pred = lr_model.predict(X_val_scaled)

**Evaluate**

*MAE:* On average, how many dollars the model is "off" by.

*R² Score:* How much of the price variance is explained by the features (0 to 1).

In [ ]:
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_val, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R-squared Score (R2): {r2:.4f}")

**Actual vs Predicted Price**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_val, y=y_pred, alpha=0.5)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], '--r', linewidth=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Actual vs. Predicted Airbnb Prices')
plt.show()

**Residual Plot**

In [ ]:
residuals = y_val - y_pred

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Price ($)')
plt.ylabel('Residuals (Errors)')
plt.title('Residual Plot (Checking for Patterns in Error)')
plt.show()

**Distribution of Errors**

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(residuals, bins=30, kde=True)
plt.title('Distribution of Residuals')
plt.xlabel('Error Amount ($)')
plt.show()

**Determinant Features**

In [ ]:
# Create a dataframe of features and their coefficients
importance = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr_model.coef_})
importance = importance.sort_values(by='Coefficient', ascending=False)

print("Top determinants of price:")
print(importance)

In [ ]:
# Final evaluation on test set
y_pred_lr_test = lr_model.predict(X_test_scaled)

mae_lr_test  = mean_absolute_error(y_test, y_pred_lr_test)
rmse_lr_test = np.sqrt(mean_squared_error(y_test, y_pred_lr_test))
r2_lr_test   = r2_score(y_test, y_pred_lr_test)

print("── Linear Regression – Test Set Results ──")
print(f"MAE  : ${mae_lr_test:.2f}")
print(f"RMSE : ${rmse_lr_test:.2f}")
print(f"R²   : {r2_lr_test:.4f}")

## Comparison: Analysis after dropping 'neighbourhood_group_Manhattan'

### Data Preprocessing for 'neighbourhood_group_Manhattan' Drop

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Re-initialize df_final_manhattan_scenario from the original df to ensure a clean slate
# The 'df' DataFrame still contains all original columns before any borough drops.
features_comp = [
    'neighbourhood_group', 'latitude', 'longitude',
    'room_type', 'price', 'minimum_nights',
    'number_of_reviews', 'calculated_host_listings_count', 'availability_365'
]
df_manhattan_scenario = df[features_comp].copy()

df_manhattan_scenario = pd.get_dummies(df_manhattan_scenario, columns=['neighbourhood_group', 'room_type'], drop_first=True)

# Drop 'neighbourhood_group_Manhattan' for this scenario
if 'neighbourhood_group_Manhattan' in df_manhattan_scenario.columns:
    df_manhattan_scenario = df_manhattan_scenario.drop(columns=['neighbourhood_group_Manhattan'])
    print("Dropped 'neighbourhood_group_Manhattan' for comparison.")
else:
    print("'neighbourhood_group_Manhattan' not found in DataFrame for comparison, no action taken.")

display(df_manhattan_scenario.head())

### Data Splitting and Scaling

In [ ]:
# 1. Define X_comp (Features) and y_comp (Target)
X_comp = df_manhattan_scenario.drop('price', axis=1)
y_comp = df_manhattan_scenario['price']

# 2. Split into Train (70%) and Temp (30%)
X_comp_train, X_comp_temp, y_comp_train, y_comp_temp = train_test_split(
    X_comp, y_comp, test_size=0.30, random_state=42, shuffle=True
)

# 3. Split Temp (30%) into Val (15%) and Test (15%)
X_comp_val, X_comp_test, y_comp_val, y_comp_test = train_test_split(
    X_comp_temp, y_comp_temp, test_size=0.50, random_state=42, shuffle=True
)

print(f"Train size: {len(X_comp_train)}, Val size: {len(X_comp_val)}, Test size: {len(X_comp_test)}")

# Initialize the scaler
scaler_comp = StandardScaler()

# Fit/Transform based on the rules learned from X_comp_train
X_comp_train_scaled = scaler_comp.fit_transform(X_comp_train)

# Use those same rules to scale Val and Test
X_comp_val_scaled = scaler_comp.transform(X_comp_val)
X_comp_test_scaled = scaler_comp.transform(X_comp_test)

print("Data is split and scaled for Manhattan drop scenario.")

### Linear Regression Model Training and Evaluation

In [ ]:
# Initialize and Train
lr_model_manhattan = LinearRegression()
lr_model_manhattan.fit(X_comp_train_scaled, y_comp_train)

# Make Predictions on the Validation set
y_comp_pred = lr_model_manhattan.predict(X_comp_val_scaled)

mae_comp = mean_absolute_error(y_comp_val, y_comp_pred)
mse_comp = mean_squared_error(y_comp_val, y_comp_pred)
rmse_comp = np.sqrt(mse_comp)
r2_comp = r2_score(y_comp_val, y_comp_pred)

print(f"Mean Absolute Error (MAE): ${mae_comp:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse_comp:.2f}")
print(f"R-squared Score (R2): {r2_comp:.4f}")

### Determinant Features (Manhattan Dropped)

In [ ]:
# Create a dataframe of features and their coefficients
importance_manhattan = pd.DataFrame({'Feature': X_comp.columns, 'Coefficient': lr_model_manhattan.coef_})
importance_manhattan = importance_manhattan.sort_values(by='Coefficient', ascending=False)

print("Top determinants of price (with Manhattan dropped):")
print(importance_manhattan)

## Comparison: Analysis with No Neighbourhood Group Features Dropped

### Data Preprocessing for No Drop Scenario

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Re-initialize df_final_no_drop from the original df to ensure a clean slate
features_no_drop = [
    'neighbourhood_group', 'latitude', 'longitude',
    'room_type', 'price', 'minimum_nights',
    'number_of_reviews', 'calculated_host_listings_count', 'availability_365'
]
df_no_drop = df[features_no_drop].copy()

df_no_drop = pd.get_dummies(df_no_drop, columns=['neighbourhood_group', 'room_type'], drop_first=True)

print("DataFrame with no neighbourhood group features dropped:")
display(df_no_drop.head())

### Data Splitting and Scaling (No Drop Scenario)

In [ ]:
# 1. Define X_no_drop (Features) and y_no_drop (Target)
X_no_drop = df_no_drop.drop('price', axis=1)
y_no_drop = df_no_drop['price']

# 2. Split into Train (70%) and Temp (30%)
X_no_drop_train, X_no_drop_temp, y_no_drop_train, y_no_drop_temp = train_test_split(
    X_no_drop, y_no_drop, test_size=0.30, random_state=42, shuffle=True
)

# 3. Split Temp (30%) into Val (15%) and Test (15%)
X_no_drop_val, X_no_drop_test, y_no_drop_val, y_no_drop_test = train_test_split(
    X_no_drop_temp, y_no_drop_temp, test_size=0.50, random_state=42, shuffle=True
)

print(f"Train size: {len(X_no_drop_train)}, Val size: {len(X_no_drop_val)}, Test size: {len(X_no_drop_test)}")

# Initialize the scaler
scaler_no_drop = StandardScaler()

# Fit/Transform based on the rules learned from X_no_drop_train
X_no_drop_train_scaled = scaler_no_drop.fit_transform(X_no_drop_train)

# Use those same rules to scale Val and Test
X_no_drop_val_scaled = scaler_no_drop.transform(X_no_drop_val)
X_no_drop_test_scaled = scaler_no_drop.transform(X_no_drop_test)

print("Data is split and scaled for no drop scenario.")

### Linear Regression Model Training and Evaluation (No Drop Scenario)

In [ ]:
# Initialize and Train
lr_model_no_drop = LinearRegression()
lr_model_no_drop.fit(X_no_drop_train_scaled, y_no_drop_train)

# Make Predictions on the Validation set
y_no_drop_pred = lr_model_no_drop.predict(X_no_drop_val_scaled)

mae_no_drop = mean_absolute_error(y_no_drop_val, y_no_drop_pred)
mse_no_drop = mean_squared_error(y_no_drop_val, y_no_drop_pred)
rmse_no_drop = np.sqrt(mse_no_drop)
r2_no_drop = r2_score(y_no_drop_val, y_no_drop_pred)

print(f"Mean Absolute Error (MAE): ${mae_no_drop:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse_no_drop:.2f}")
print(f"R-squared Score (R2): {r2_no_drop:.4f}")

### Determinant Features (No Drop Scenario)

In [ ]:
# Create a dataframe of features and their coefficients
importance_no_drop = pd.DataFrame({'Feature': X_no_drop.columns, 'Coefficient': lr_model_no_drop.coef_})
importance_no_drop = importance_no_drop.sort_values(by='Coefficient', ascending=False)

print("Top determinants of price (no neighbourhood group features dropped):")
print(importance_no_drop)

Gemini's Comments
1. Scenario: 'neighbourhood_group_Brooklyn' Dropped

Mean Absolute Error (MAE): $50.63 * **Root Mean Squared Error (RMSE):** $$50.63 * **Root Mean Squared Error (RMSE):** $79.02
R-squared Score (R2): 0.3797
Top determinants (positive): 'neighbourhood_group_Manhattan', 'availability_365'
Top determinants (negative): 'room_type_Private room', 'room_type_Shared room'
2. Scenario: 'neighbourhood_group_Manhattan' Dropped

Mean Absolute Error (MAE): $52.26 * **Root Mean Squared Error (RMSE):** $$52.26 * **Root Mean Squared Error (RMSE):** $81.73
R-squared Score (R2): 0.3494
Top determinants (positive): 'availability_365', 'calculated_host_listings_count'
Top determinants (negative): 'room_type_Private room', 'neighbourhood_group_Brooklyn'
3. Scenario: No Neighbourhood Group Features Dropped (New Results)

Mean Absolute Error (MAE): $52.16 * **Root Mean Squared Error (RMSE):** $$52.16 * **Root Mean Squared Error (RMSE):** $81.65
R-squared Score (R2): 0.3506
Top determinants (positive): 'availability_365', 'neighbourhood_group_Manhattan'
Top determinants (negative): 'room_type_Private room', 'room_type_Shared room', 'longitude'
Overall Comparison and Insights:

Best Performance: The model performed best when 'neighbourhood_group_Brooklyn' was dropped, yielding the lowest MAE and RMSE, and the highest R-squared score (0.3797). This indicates that this configuration provided the most accurate predictions and explained the most variance in price.
Impact of Dropping Manhattan: When 'neighbourhood_group_Manhattan' was dropped, the model's performance decreased significantly (R2 dropped to 0.3494), confirming its importance as a predictor. In this case, 'availability_365' took over as the most influential positive feature.
Impact of Dropping Neither (Keeping All Features): This scenario produced results very similar to when 'neighbourhood_group_Manhattan' was dropped (R2 of 0.3506). The performance is slightly better than dropping Manhattan, but still worse than dropping Brooklyn. This suggests that the high correlation between 'neighbourhood_group_Manhattan' and 'neighbourhood_group_Brooklyn' does have an effect, and explicitly removing one of them (especially 'Brooklyn' in this case) helps the model disentangle their individual contributions better, or at least reduces noise that the model might struggle with due to multicollinearity.
Conclusion:

While 'neighbourhood_group_Manhattan' and 'neighbourhood_group_Brooklyn' are highly correlated, removing 'neighbourhood_group_Brooklyn' led to the strongest model performance among the three scenarios. This suggests that 'neighbourhood_group_Manhattan' is a more valuable feature for predicting Airbnb prices, and handling its correlation by dropping the less impactful 'Brooklyn' feature is beneficial. The difference in performance between 'dropping Manhattan' and 'dropping neither' is minimal, reinforcing the idea that having both highly correlated features might not add much predictive power over just having the more dominant one.

This comprehensive analysis provides a clear picture of how managing correlated features, even subtly, can impact your model's accuracy.

## Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

### Model Training (Brooklyn Dropped Scenario)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import numpy as np
import os

# 0. Ensure base data is downloaded and loaded
try:
    if not os.path.exists('nyc_data/AB_NYC_2019.csv'):
        print('Downloading dataset...')
        os.environ['KAGGLE_API_TOKEN'] =  "YOUR_KAGGLE_TOKEN_HERE"
        !kaggle datasets download -d dgomonov/new-york-city-airbnb-open-data
        !unzip -o new-york-city-airbnb-open-data.zip -d nyc_data

    df = pd.read_csv('nyc_data/AB_NYC_2019.csv')
    # Apply the same basic cleaning (dropping nulls and price outliers) as done previously
    df.drop(['last_review', 'reviews_per_month'], axis=1, inplace=True)
    df.dropna(inplace=True)
    df = df[df['price'] <= 795]
except Exception as e:
    print(f'Error setup data: {e}')

# 1. Re-create df_final from the original df
features = [
    'neighbourhood_group', 'latitude', 'longitude',
    'room_type', 'price', 'minimum_nights',
    'number_of_reviews', 'calculated_host_listings_count', 'availability_365'
]
df_final = df[features].copy()

# 2. Preprocess: Encoding and Outlier Removal (Minimum Nights)
df_final = pd.get_dummies(df_final, columns=['neighbourhood_group', 'room_type'], drop_first=True)
min_nights_99 = df_final['minimum_nights'].quantile(0.99)
df_final = df_final[df_final['minimum_nights'] <= min_nights_99]

# 3. Handle Multicollinearity: Drop Brooklyn
if 'neighbourhood_group_Brooklyn' in df_final.columns:
    df_final = df_final.drop(columns=['neighbourhood_group_Brooklyn'])

# 4. Define X and y
X = df_final.drop('price', axis=1)
y = df_final['price']

# 5. Split and Scale
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 6. Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
print('Random Forest model trained successfully after restoring data!')

### Validation Results

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred_rf = rf_model.predict(X_val_scaled)

mae_rf  = mean_absolute_error(y_val, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
r2_rf   = r2_score(y_val, y_pred_rf)

print("── Random Forest (Brooklyn Dropped) – Validation Results ──")
print(f"MAE  : ${mae_rf:.2f}")
print(f"RMSE : ${rmse_rf:.2f}")
print(f"R²   : {r2_rf:.4f}")

### Comparison with Linear Regression Baseline

In [ ]:
# Compare Random Forest against the Linear Regression baseline
print("=" * 45)
print(f"{'Model':<25} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 45)
print(f"{'Linear Regression':<25} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest':<25} ${mae_rf:<5.2f}  ${rmse_rf:<6.2f}  {r2_rf:<6.4f}")
print("=" * 45)

### Feature Importance Analysis

In [ ]:
import matplotlib.pyplot as plt

# Extract feature names from the original (unscaled) DataFrame
feature_names = X_train.columns

importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feat_imp_df)

# Plot feature importances as a horizontal bar chart
plt.figure(figsize=(10, 5))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.title('Random Forest – Feature Importances')
plt.tight_layout()
plt.show()

### Actual vs. Predicted Plot

In [ ]:
import seaborn as sns

# Scatter plot of actual vs predicted prices
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_pred_rf, alpha=0.3, color='steelblue', label='Predicted')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()],
         'r--', label='Perfect Prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Random Forest – Actual vs. Predicted Airbnb Prices')
plt.legend()
plt.tight_layout()
plt.show()

### Residual Analysis

In [ ]:
# Calculate residuals (errors) between actual and predicted values
residuals_rf = y_val - y_pred_rf

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_rf, residuals_rf, alpha=0.3, color='steelblue')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted Price ($)')
plt.ylabel('Residuals (Errors)')
plt.title('Random Forest – Residual Plot')
plt.tight_layout()
plt.show()

### Distribution of Residuals

In [ ]:
import seaborn as sns

# Distribution of residuals
plt.figure(figsize=(8, 5))
plt.hist(residuals_rf, bins=50, color='steelblue', edgecolor='white')
sns.kdeplot(residuals_rf, color='navy')
plt.axvline(0, color='red', linestyle='--')
plt.xlabel('Error Amount ($)')
plt.ylabel('Count')
plt.title('Random Forest – Distribution of Residuals')
plt.tight_layout()
plt.show()

### Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Define the parameter grid to search over
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Initialize RandomizedSearchCV
rf_random = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,          # try 20 random combinations
    cv=3,               # 3-fold cross-validation
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=2           # print progress
)

rf_random.fit(X_train_scaled, y_train)

print("\nBest Parameters Found:")
print(rf_random.best_params_)

### Tuned Random Forest Model

In [ ]:
# Train a new Random Forest model with the best parameters found
rf_tuned = RandomForestRegressor(
    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features='log2',
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train_scaled, y_train)

# Evaluate on validation set
y_pred_rf_tuned = rf_tuned.predict(X_val_scaled)

mae_tuned  = mean_absolute_error(y_val, y_pred_rf_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_val, y_pred_rf_tuned))
r2_tuned   = r2_score(y_val, y_pred_rf_tuned)

# Comparison table
print("=" * 55)
print(f"{'Model':<30} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 55)
print(f"{'Linear Regression':<30} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest (default)':<30} {'$44.92':>6}  {'$72.63':>7}  {'0.4760':>6}")
print(f"{'Random Forest (tuned)':<30} ${mae_tuned:<5.2f}  ${rmse_tuned:<6.2f}  {r2_tuned:<6.4f}")
print("=" * 55)

### Tuned Model – Visual Evaluation

In [ ]:
residuals_tuned = y_val - y_pred_rf_tuned

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_val, y_pred_rf_tuned, alpha=0.3, color='steelblue')
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Tuned RF – Actual vs. Predicted')

# 2. Residual Plot
axes[1].scatter(y_pred_rf_tuned, residuals_tuned, alpha=0.3, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residuals (Errors)')
axes[1].set_title('Tuned RF – Residual Plot')

# 3. Distribution of Residuals
axes[2].hist(residuals_tuned, bins=50, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Error Amount ($)')
axes[2].set_ylabel('Count')
axes[2].set_title('Tuned RF – Distribution of Residuals')

plt.tight_layout()
plt.show()

### Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

# 5-fold cross-validation on the tuned model using the training data
cv_scores = cross_val_score(
    rf_tuned,
    X_train_scaled,
    y_train,
    cv=5,                        # 5-fold
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print("Cross-Validation R² Scores (5-fold):")
print([f"{s:.4f}" for s in cv_scores])
print(f"\nMean R²  : {cv_scores.mean():.4f}")
print(f"Std Dev  : {cv_scores.std():.4f}")

### Model Comparison: Baseline vs. Optimized Random Forest

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Data for comparison: Baseline vs Tuned
models = ['Linear Regression (Baseline)', 'Random Forest (Optimized)']
r2_scores = [0.3797, 0.5092]
mae_values = [50.65, 43.09]

# Create a comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': models,
    'R2 Score': r2_scores,
    'MAE ($)': mae_values
})

# Set up the figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: R-squared Comparison
sns.barplot(x='Model', y='R2 Score', data=comparison_df, palette='viridis', hue='Model', legend=False, ax=ax1)
ax1.set_title('R² Score Comparison (Higher is Better)', fontsize=14)
ax1.set_ylim(0, 0.6)
for i, v in enumerate(r2_scores):
    ax1.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

# Plot 2: MAE Comparison
sns.barplot(x='Model', y='MAE ($)', data=comparison_df, palette='magma', hue='Model', legend=False, ax=ax2)
ax2.set_title('MAE Comparison (Lower is Better)', fontsize=14)
ax2.set_ylabel('Mean Absolute Error ($)')
for i, v in enumerate(mae_values):
    ax2.text(i, v + 0.5, f'${v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## FINAL MODEL EVALUATION (TEST SET)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Scale the test features using the same scaler fitted on training data
X_test_scaled = scaler.transform(X_test)

# 2. Make predictions using the tuned Random Forest model
y_test_pred = rf_tuned.predict(X_test_scaled)

# 3. Calculate metrics
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print("── Final Random Forest (Optimized) – Test Set Results ──")
print(f"MAE  : ${mae_test:.2f}")
print(f"RMSE : ${rmse_test:.2f}")
print(f"R%²   : {r2_test:.4f}")

# 4. Final Verification Comparison
print("\nComparison check:")
print(f"Validation R%²: {r2_tuned:.4f}")
print(f"Test R%²      : {r2_test:.4f}")

## Executive Summary: Random Forest Model for NYC Airbnb Price Prediction

### 1. Project Overview
Following the baseline evaluation using Linear Regression, we implemented a **Random Forest Regressor** to capture non-linear relationships and improve predictive accuracy for NYC Airbnb prices.

### 2. Model Development & Optimization
*   **Handling Multicollinearity:** Based on earlier analysis, the 'Brooklyn' neighborhood group was dropped to address high correlation with 'Manhattan', allowing the model to focus on more distinct price drivers.
*   **Feature Engineering:** Features included geographical coordinates, room types (One-Hot Encoded), and listing attributes like minimum nights and availability.
*   **Hyperparameter Tuning:** We utilized `RandomizedSearchCV` to optimize the model. The best-performing parameters were:
    *   `n_estimators`: 300
    *   `max_depth`: 20
    *   `max_features`: 'log2'
    *   `min_samples_leaf`: 2

### 3. Performance Results
The transition from a linear to an ensemble-based model yielded significant improvements:

| Metric | Linear Regression (Baseline) | Random Forest (Optimized) |
| :--- | :--- | :--- |
| **R² Score** | 0.3797 | **0.5027** |
| **MAE** | $50.65 | **$44.13** |
| **RMSE** | $79.03 | **$74.12** |

### 4. Key Findings
*   **Generalization:** The model showed excellent consistency between validation ($R^2=0.5092$) and test ($R^2=0.5027$) sets, indicating no significant overfitting.
*   **Feature Importance:** The most influential predictors were **Room Type (Private Room)** and **Geographical Location (Longitude/Latitude)**, followed by listing availability.
*   **Residual Analysis:** While the model performs robustly for standard listings, residual plots indicate higher variance and underestimation for high-end luxury listings (prices >$400).

### 5. Conclusion
The Optimized Random Forest model successfully explains approximately **50% of the price variance**, providing a reliable framework for price estimation. Future iterations could explore Gradient Boosting (XGBoost/LightGBM) or advanced feature engineering (e.g., proximity to landmarks) for further gains.

## XGBoost Model

In [ ]:
from xgboost import XGBRegressor

# Train XGBoost with default parameters as baseline
xgb_model = XGBRegressor(
    n_estimators=100,     # number of boosting rounds
    learning_rate=0.1,    # step size shrinkage to prevent overfitting
    max_depth=6,          # maximum depth of each tree
    random_state=42,
    n_jobs=-1,
    verbosity=0           # suppress training logs
)

xgb_model.fit(X_train_scaled, y_train)
print("XGBoost model trained successfully!")

### Validation Results

In [ ]:
# Evaluate XGBoost on the validation set
y_pred_xgb = xgb_model.predict(X_val_scaled)

mae_xgb  = mean_absolute_error(y_val, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
r2_xgb   = r2_score(y_val, y_pred_xgb)

print("── XGBoost (Default) – Validation Results ──")
print(f"MAE  : ${mae_xgb:.2f}")
print(f"RMSE : ${rmse_xgb:.2f}")
print(f"R²   : {r2_xgb:.4f}")

# Comparison with previous models
print("\n" + "=" * 55)
print(f"{'Model':<30} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 55)
print(f"{'Linear Regression':<30} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest (tuned)':<30} {'$43.09':>6}  {'$70.29':>7}  {'0.5092':>6}")
print(f"{'XGBoost (default)':<30} ${mae_xgb:<5.2f}  ${rmse_xgb:<6.2f}  {r2_xgb:<6.4f}")
print("=" * 55)

### Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter grid
param_dist_xgb = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 6, 8, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

xgb_random = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    param_distributions=param_dist_xgb,
    n_iter=20,
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

xgb_random.fit(X_train_scaled, y_train)

print("\nBest Parameters Found:")
print(xgb_random.best_params_)

### Tuned XGBoost Model

In [ ]:
# Train XGBoost with the best parameters found
xgb_tuned = XGBRegressor(
    subsample=0.8,
    n_estimators=300,
    min_child_weight=3,
    max_depth=5,
    learning_rate=0.05,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_tuned.fit(X_train_scaled, y_train)

# Evaluate on validation set
y_pred_xgb_tuned = xgb_tuned.predict(X_val_scaled)

mae_xgb_tuned  = mean_absolute_error(y_val, y_pred_xgb_tuned)
rmse_xgb_tuned = np.sqrt(mean_squared_error(y_val, y_pred_xgb_tuned))
r2_xgb_tuned   = r2_score(y_val, y_pred_xgb_tuned)

# Comparison table
print("=" * 55)
print(f"{'Model':<30} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 55)
print(f"{'Linear Regression':<30} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest (tuned)':<30} {'$43.09':>6}  {'$70.29':>7}  {'0.5092':>6}")
print(f"{'XGBoost (default)':<30} {'$43.88':>6}  {'$71.04':>7}  {'0.4987':>6}")
print(f"{'XGBoost (tuned)':<30} ${mae_xgb_tuned:<5.2f}  ${rmse_xgb_tuned:<6.2f}  {r2_xgb_tuned:<6.4f}")
print("=" * 55)

### Tuned XGBoost – Visual Evaluation

In [ ]:
residuals_xgb = y_val - y_pred_xgb_tuned

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_val, y_pred_xgb_tuned, alpha=0.3, color='darkorange')
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('XGBoost (Tuned) – Actual vs. Predicted')

# 2. Residual Plot
axes[1].scatter(y_pred_xgb_tuned, residuals_xgb, alpha=0.3, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residuals (Errors)')
axes[1].set_title('XGBoost (Tuned) – Residual Plot')

# 3. Distribution of Residuals
axes[2].hist(residuals_xgb, bins=50, color='darkorange', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Error Amount ($)')
axes[2].set_ylabel('Count')
axes[2].set_title('XGBoost (Tuned) – Distribution of Residuals')

plt.tight_layout()
plt.show()

### Feature Importance Analysis

In [ ]:
# Extract and plot feature importances from the tuned XGBoost model
feat_imp_xgb = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_tuned.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feat_imp_xgb)

plt.figure(figsize=(10, 5))
plt.barh(feat_imp_xgb['Feature'], feat_imp_xgb['Importance'], color='darkorange')
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.title('XGBoost (Tuned) – Feature Importances')
plt.tight_layout()
plt.show()

### Final Evaluation on Test Set

In [ ]:
# Evaluate tuned XGBoost on the unseen test set
y_pred_xgb_test = xgb_tuned.predict(X_test_scaled)

mae_xgb_test  = mean_absolute_error(y_test, y_pred_xgb_test)
rmse_xgb_test = np.sqrt(mean_squared_error(y_test, y_pred_xgb_test))
r2_xgb_test   = r2_score(y_test, y_pred_xgb_test)

print("── XGBoost (Tuned) – Test Set Results ──")
print(f"MAE  : ${mae_xgb_test:.2f}")
print(f"RMSE : ${rmse_xgb_test:.2f}")
print(f"R²   : {r2_xgb_test:.4f}")

print(f"\nComparison check:")
print(f"Validation R²: {r2_xgb_tuned:.4f}")
print(f"Test R²      : {r2_xgb_test:.4f}")

### Feature Importance Comparison: Random Forest vs. XGBoost

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Prepare data for comparison
fi_rf = pd.DataFrame({'Feature': X_train.columns, 'Importance_RF': rf_tuned.feature_importances_})
fi_xgb = pd.DataFrame({'Feature': X_train.columns, 'Importance_XGB': xgb_tuned.feature_importances_})

# Merge and melt for plotting
fi_comp = pd.merge(fi_rf, fi_xgb, on='Feature').melt(id_vars='Feature', var_name='Model', value_name='Importance')

# Plot
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', hue='Model', data=fi_comp, palette=['steelblue', 'darkorange'])
plt.title('Feature Importance: Random Forest vs. XGBoost')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### XGBoost Learning Curve Analysis
To understand if our XGBoost model is overfitting or could benefit from more data, we plot the learning curve.

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, val_scores = learning_curve(
    xgb_tuned, X_train_scaled, y_train, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5)
)

# Calculate mean and std
train_mean = -np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = -np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training MAE")
plt.plot(train_sizes, val_mean, 'o-', color="g", label="Cross-validation MAE")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color="g")

plt.title("XGBoost Learning Curve")
plt.xlabel("Training Set Size")
plt.ylabel("MAE ($)")
plt.legend(loc="best")
plt.grid(True)
plt.show()

### Recommendations for Improving the XGBoost Model

Based on the learning curve and performance metrics, the following steps can be taken to address XGBoost's slight underperformance compared to Random Forest and its tendency toward mild overfitting:

1. **Increase Regularization Parameters:**
   - `gamma`: Controls the minimum loss reduction required to make a further partition on a leaf node. Increasing this makes the model more conservative.
   - `lambda` (L2) and `alpha` (L1): Increasing the penalty on weights limits model complexity.

2. **Reduce Tree Depth (`max_depth`):**
   - Currently, `max_depth` is around 5 or 6. If the training error is very low but the validation error is high, reducing the depth to 3 or 4 can improve generalization.

3. **Feature Engineering:**
   - **Coordinate Clustering:** Instead of using latitude and longitude as raw numeric values, they can be grouped using 'K-Means' to better capture neighborhood effects.
   - **Price Log Transformation:** Since the price distribution is right-skewed, training on `log1p(price)` and then transforming back may help XGBoost learn more stably.

4. **Use Early Stopping:**
   - Instead of a fixed `n_estimators`, the `early_stopping_rounds` parameter should be activated to stop training when the validation error no longer decreases. This prevents overfitting from the start.

5. **Broader Hyperparameter Search:**
   - Lowering the `learning_rate` further (e.g., to 0.01) while increasing the number of `n_estimators` can help the model learn finer details.

## Executive Summary: XGBoost Model for NYC Airbnb Price Prediction

### 1. Objective
The goal was to implement and optimize an **XGBoost (Extreme Gradient Boosting)** regressor to further enhance the predictive accuracy for NYC Airbnb prices, following the baseline and Random Forest implementations.

### 2. Development Process
*   **Environment Setup:** The model was trained using the `xgboost` library, utilizing scaled features from the preprocessed NYC Airbnb dataset.
*   **Feature Engineering Consistency:** We maintained the same feature set as the Random Forest model (dropping 'Brooklyn' to mitigate multicollinearity) to ensure a fair performance comparison.
*   **Initial Baseline:** An initial XGBoost model with default parameters (`max_depth=6`, `learning_rate=0.1`) was established to set a performance floor.

### 3. Hyperparameter Optimization
We conducted a `RandomizedSearchCV` across a broad parameter grid to find the most effective configuration. The search focused on:
*   `n_estimators`: 300
*   `max_depth`: 5
*   `learning_rate`: 0.05
*   `subsample` & `colsample_bytree`: 0.8 (to prevent overfitting)
*   `min_child_weight`: 3

### 4. Performance Metrics (Validation vs. Test)
| Scenario | MAE | RMSE | R² Score |
| :--- | :--- | :--- | :--- |
| **Validation (Tuned)** | $43.72 | $70.63 | 0.5044 |
| **Final Test Set** | **$44.89** | **$74.83** | **0.4931** |

### 5. Key Insights & Comparisons
*   **Stability:** The model showed consistent results between validation and test sets, suggesting reliable generalization.
*   **Feature Importance:** XGBoost placed extremely high weight on **Room Type (Private Room vs. Shared Room)** and **Borough Location (Manhattan)** as the primary drivers of price.
*   **Relative Performance:** While highly accurate, the tuned XGBoost performed slightly behind the Optimized Random Forest (Test R²: 0.5027 vs 0.4931). Learning curve analysis suggests XGBoost was more prone to slight overfitting on specific features compared to the ensemble approach of Random Forest.

### 6. Final Verdict
The XGBoost model provides a robust alternative that explains approximately **49-50% of the price variance**. Its speed and interpretability regarding categorical features (like room types) make it a valuable component of the overall analysis.

## MLP Model (PyTorch)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Create DataLoaders for batch training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Input features : {X_train_tensor.shape[1]}")
print(f"Training samples: {X_train_tensor.shape[0]}")
print("Tensors created successfully!")

### Model Architecture

In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressor, self).__init__()
        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Second hidden layer
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Third hidden layer
            nn.Linear(64, 32),
            nn.ReLU(),

            # Output layer (single price prediction)
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# Initialize the model
input_dim = X_train_tensor.shape[1]  # 11 features
mlp_model = MLPRegressor(input_dim)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

print(mlp_model)
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_model.parameters())}")

### Model Training

In [ ]:
# Training configuration
num_epochs = 100
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # ── Training phase ──
    mlp_model.train()
    batch_losses = []

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()           # clear gradients
        predictions = mlp_model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()                 # backpropagation
        optimizer.step()                # update weights
        batch_losses.append(loss.item())

    train_loss = sum(batch_losses) / len(batch_losses)
    train_losses.append(train_loss)

    # ── Validation phase ──
    mlp_model.eval()
    with torch.no_grad():
        val_predictions = mlp_model(X_val_tensor)
        val_loss = criterion(val_predictions, y_val_tensor)
        val_losses.append(val_loss.item())

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.2f} | "
              f"Val Loss: {val_loss.item():.2f}")

print("\nTraining complete!")

### Validation Results

In [ ]:
mlp_model.eval()
with torch.no_grad():
    y_pred_mlp = mlp_model(X_val_tensor).numpy().flatten()

mae_mlp  = mean_absolute_error(y_val, y_pred_mlp)
rmse_mlp = np.sqrt(mean_squared_error(y_val, y_pred_mlp))
r2_mlp   = r2_score(y_val, y_pred_mlp)

print("── MLP – Validation Results ──")
print(f"MAE  : ${mae_mlp:.2f}")
print(f"RMSE : ${rmse_mlp:.2f}")
print(f"R²   : {r2_mlp:.4f}")

# Comparison with all models
print("\n" + "=" * 55)
print(f"{'Model':<30} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 55)
print(f"{'Linear Regression':<30} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest (tuned)':<30} {'$43.09':>6}  {'$70.29':>7}  {'0.5092':>6}")
print(f"{'XGBoost (tuned)':<30} {'$43.72':>6}  {'$70.63':>7}  {'0.5044':>6}")
print(f"{'MLP':<30} ${mae_mlp:<5.2f}  ${rmse_mlp:<6.2f}  {r2_mlp:<6.4f}")
print("=" * 55)

### Improved Training (200 Epochs)

In [ ]:
# Re-initialize model with lower learning rate
mlp_model2 = MLPRegressor(input_dim)
optimizer2 = torch.optim.Adam(mlp_model2.parameters(), lr=0.0005)

num_epochs = 200
train_losses2 = []
val_losses2 = []

for epoch in range(num_epochs):
    # Training phase
    mlp_model2.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer2.zero_grad()
        predictions = mlp_model2(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer2.step()
        batch_losses.append(loss.item())

    train_loss = sum(batch_losses) / len(batch_losses)
    train_losses2.append(train_loss)

    mlp_model2.eval()
    with torch.no_grad():
        val_predictions = mlp_model2(X_val_tensor)
        val_loss = criterion(val_predictions, y_val_tensor)
        val_losses2.append(val_loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.2f} | "
              f"Val Loss: {val_loss.item():.2f}")

print("\nTraining complete!")

### Improved Model – Validation Results

In [ ]:
mlp_model2.eval()
with torch.no_grad():
    y_pred_mlp2 = mlp_model2(X_val_tensor).numpy().flatten()

mae_mlp2  = mean_absolute_error(y_val, y_pred_mlp2)
rmse_mlp2 = np.sqrt(mean_squared_error(y_val, y_pred_mlp2))
r2_mlp2   = r2_score(y_val, y_pred_mlp2)

print("── MLP (200 Epochs) – Validation Results ──")
print(f"MAE  : ${mae_mlp2:.2f}")
print(f"RMSE : ${rmse_mlp2:.2f}")
print(f"R²   : {r2_mlp2:.4f}")

print("\n" + "=" * 55)
print(f"{'Model':<30} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 55)
print(f"{'Linear Regression':<30} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest (tuned)':<30} {'$43.09':>6}  {'$70.29':>7}  {'0.5092':>6}")
print(f"{'XGBoost (tuned)':<30} {'$43.72':>6}  {'$70.63':>7}  {'0.5044':>6}")
print(f"{'MLP (100 epochs)':<30} {'$44.08':>6}  {'$75.18':>7}  {'0.4386':>6}")
print(f"{'MLP (200 epochs)':<30} ${mae_mlp2:<5.2f}  ${rmse_mlp2:<6.2f}  {r2_mlp2:<6.4f}")
print("=" * 55)

### Extended Training (300 Epochs)
Let's push the MLP model further to 300 epochs with a slightly adjusted learning rate to see if we can converge to a better R² score.

In [ ]:
import matplotlib.pyplot as plt

# Re-initialize for a fresh longer run
mlp_model3 = MLPRegressor(input_dim)
optimizer3 = torch.optim.Adam(mlp_model3.parameters(), lr=0.0003) # Slightly lower LR for stability

num_epochs = 300
train_losses3 = []
val_losses3 = []

print("Starting extended training for 300 epochs...")

for epoch in range(num_epochs):
    mlp_model3.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer3.zero_grad()
        predictions = mlp_model3(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer3.step()
        batch_losses.append(loss.item())

    train_loss = sum(batch_losses) / len(batch_losses)
    train_losses3.append(train_loss)

    mlp_model3.eval()
    with torch.no_grad():
        val_predictions = mlp_model3(X_val_tensor)
        v_loss = criterion(val_predictions, y_val_tensor)
        val_losses3.append(v_loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.2f} | Val Loss: {v_loss.item():.2f}")

# Visualize Loss History
plt.figure(figsize=(10, 5))
plt.plot(train_losses3, label='Train Loss')
plt.plot(val_losses3, label='Val Loss')
plt.title('MLP Training Loss History (300 Epochs)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

# Final Val Metrics for this run
mlp_model3.eval()
with torch.no_grad():
    y_pred_mlp3 = mlp_model3(X_val_tensor).numpy().flatten()

mae_mlp3 = mean_absolute_error(y_val, y_pred_mlp3)
r2_mlp3 = r2_score(y_val, y_pred_mlp3)

print(f"\n── MLP (300 Epochs) Results ──")
print(f"MAE : ${mae_mlp3:.2f}")
print(f"R²  : {r2_mlp3:.4f}")

### Improved MLP Architecture (Batch Normalization)

In [ ]:
class MLPRegressorV2(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressorV2, self).__init__()
        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Second hidden layer
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Third hidden layer
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),

            # Fourth hidden layer
            nn.Linear(64, 32),
            nn.ReLU(),

            # Output layer
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# Initialize improved model
mlp_v2 = MLPRegressorV2(input_dim)
optimizer_v2 = torch.optim.Adam(mlp_v2.parameters(), lr=0.001)

print(mlp_v2)
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_v2.parameters())}")

### Training

In [ ]:
num_epochs = 200
train_losses_v2 = []
val_losses_v2 = []

for epoch in range(num_epochs):
    # Training phase
    mlp_v2.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer_v2.zero_grad()
        predictions = mlp_v2(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer_v2.step()
        batch_losses.append(loss.item())

    train_loss = sum(batch_losses) / len(batch_losses)
    train_losses_v2.append(train_loss)

    # Validation phase
    mlp_v2.eval()
    with torch.no_grad():
        val_predictions = mlp_v2(X_val_tensor)
        val_loss = criterion(val_predictions, y_val_tensor)
        val_losses_v2.append(val_loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.2f} | "
              f"Val Loss: {val_loss.item():.2f}")

print("\nTraining complete!")

# Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses_v2, label='Train Loss')
plt.plot(val_losses_v2, label='Val Loss')
plt.title('MLP V2 (Batch Norm) – Training Loss History')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.show()

### Validation Results

In [ ]:
mlp_v2.eval()
with torch.no_grad():
    y_pred_mlp_v2 = mlp_v2(X_val_tensor).numpy().flatten()

mae_mlp_v2  = mean_absolute_error(y_val, y_pred_mlp_v2)
rmse_mlp_v2 = np.sqrt(mean_squared_error(y_val, y_pred_mlp_v2))
r2_mlp_v2   = r2_score(y_val, y_pred_mlp_v2)

print("── MLP V2 (Batch Norm) – Validation Results ──")
print(f"MAE  : ${mae_mlp_v2:.2f}")
print(f"RMSE : ${rmse_mlp_v2:.2f}")
print(f"R²   : {r2_mlp_v2:.4f}")

print("\n" + "=" * 55)
print(f"{'Model':<30} {'MAE':>6}  {'RMSE':>7}  {'R²':>6}")
print("=" * 55)
print(f"{'Linear Regression':<30} {'$50.65':>6}  {'$79.03':>7}  {'0.3797':>6}")
print(f"{'Random Forest (tuned)':<30} {'$43.09':>6}  {'$70.29':>7}  {'0.5092':>6}")
print(f"{'XGBoost (tuned)':<30} {'$43.72':>6}  {'$70.63':>7}  {'0.5044':>6}")
print(f"{'MLP (200 epochs)':<30} {'$44.11':>6}  {'$73.08':>7}  {'0.4695':>6}")
print(f"{'MLP V2 (Batch Norm)':<30} ${mae_mlp_v2:<5.2f}  ${rmse_mlp_v2:<6.2f}  {r2_mlp_v2:<6.4f}")
print("=" * 55)

### Final Evaluation on Test Set

In [ ]:
# Evaluate MLP V2 on the unseen test set
mlp_v2.eval()
with torch.no_grad():
    y_pred_mlp_v2_test = mlp_v2(X_test_tensor).numpy().flatten()

mae_mlp_v2_test  = mean_absolute_error(y_test, y_pred_mlp_v2_test)
rmse_mlp_v2_test = np.sqrt(mean_squared_error(y_test, y_pred_mlp_v2_test))
r2_mlp_v2_test   = r2_score(y_test, y_pred_mlp_v2_test)

print("── MLP V2 (Batch Norm) – Test Set Results ──")
print(f"MAE  : ${mae_mlp_v2_test:.2f}")
print(f"RMSE : ${rmse_mlp_v2_test:.2f}")
print(f"R²   : {r2_mlp_v2_test:.4f}")

print(f"\nComparison check:")
print(f"Validation R²: {r2_mlp_v2:.4f}")
print(f"Test R²      : {r2_mlp_v2_test:.4f}")

## Executive Summary: MLP (Deep Learning) Model for NYC Airbnb Price Prediction

### 1. Objective
The goal was to implement a **Multi-Layer Perceptron (MLP)** using **PyTorch** to explore whether a Deep Learning approach could outperform tree-based ensemble models on the NYC Airbnb tabular dataset.

### 2. Model Evolution & Architecture
We iteratively refined the neural network to improve convergence and predictive power:
*   **Initial MLP (Baseline):** A simple 3-layer architecture (128-64-32 units) with ReLU activation and Dropout. Trained for 100-300 epochs, it established a baseline $R^2$ of ~0.44-0.45.
*   **MLP V2 (Optimized):** To improve stability and performance, we upgraded the architecture to a deeper 4-layer structure (256-128-64-32 units) and integrated **Batch Normalization** after each linear layer. This significantly smoothed the loss curve and accelerated learning.

### 3. Training Strategy
*   **Loss Function:** Mean Squared Error (MSE).
*   **Optimizer:** Adam optimizer with a dynamic learning rate (tuned from 0.001 to 0.0003 for stability).
*   **Regularization:** Dropout layers (30%, 20%, 10%) were used to prevent overfitting, ensuring the model generalized well to unseen data.

### 4. Performance Results (MLP V2)
| Metric | Validation Set | Final Test Set |
| :--- | :--- | :--- |
| **R² Score** | 0.4878 | **0.4717** |
| **MAE** | $44.01 | **$45.54** |
| **RMSE** | $71.81 | **$76.40** |

### 5. Key Findings
*   **Convergence:** The inclusion of Batch Normalization was critical. It allowed the model to reach a higher $R^2$ compared to the basic MLP by stabilizing the hidden layer distributions.
*   **Deep Learning vs. Trees:** While the MLP performed robustly and showed high consistency between validation and test sets (no overfitting), it slightly underperformed compared to the **Random Forest (Test $R^2$: 0.5027)**. This confirms that for this specific tabular dataset, tree-based models remain slightly more efficient.
*   **Scalability:** The MLP demonstrates that with further feature engineering (e.g., embedding layers for categorical data), Deep Learning can be a competitive alternative for large-scale price estimation tasks.

### 6. Conclusion
The Optimized MLP model successfully explains approximately **47-49% of the price variance**. While the Random Forest remains the 'champion' model for this project, the MLP provides a strong, non-linear alternative that generalizes exceptionally well across different NYC boroughs.

## Final Model Comparison

In [ ]:
# Final comparison of all models on test set
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
models = ['Linear\nRegression', 'Random Forest\n(tuned)',
          'XGBoost\n(tuned)', 'MLP V2\n(Batch Norm)']
mae_scores  = [53.00, 44.13, 44.89, 45.54]
rmse_scores = [83.76, 74.12, 74.83, 76.40]
r2_scores   = [0.3650, 0.5027, 0.4931, 0.4717]
colors = ['#5B9BD5', '#70AD47', '#FFC000', '#FF6B6B']

# R² Score comparison
axes[0].bar(models, r2_scores, color=colors, edgecolor='white')
axes[0].set_title('R² Score Comparison (Test Set)', fontsize=13)
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 0.65)
for i, v in enumerate(r2_scores):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=10)

# RMSE comparison
axes[1].bar(models, rmse_scores, color=colors, edgecolor='white')
axes[1].set_title('RMSE Comparison (Test Set)', fontsize=13)
axes[1].set_ylabel('RMSE ($)')
axes[1].set_ylim(0, 100)
for i, v in enumerate(rmse_scores):
    axes[1].text(i, v + 0.5, f'${v:.2f}', ha='center', fontsize=10)

plt.suptitle('NYC Airbnb Price Prediction – Model Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Final summary table
print("=" * 60)
print(f"{'Model':<25} {'MAE':>7}  {'RMSE':>7}  {'R² (Test)':>9}")
print("=" * 60)
print(f"{'Linear Regression':<25} {'$53.00':>7}  {'$83.76':>7}  {'0.3650':>9}")
print(f"{'Random Forest (tuned)':<25} {'$44.13':>7}  {'$74.12':>7}  {'0.5027':>9}")
print(f"{'XGBoost (tuned)':<25} {'$44.89':>7}  {'$74.83':>7}  {'0.4931':>9}")
print(f"{'MLP V2 (Batch Norm)':<25} {'$45.54':>7}  {'$76.40':>7}  {'0.4717':>9}")
print("=" * 60)
print(f"\n✓ Best Model: Random Forest (tuned) with R² = 0.5027")

### **Final Model Selection Summary**

In this project, we evaluated four different modeling approaches to predict NYC Airbnb prices. Here is the final comparison of their performance on the test set:

| Model | MAE | RMSE | R² Score |
| :--- | :--- | :--- | :--- |
| **Linear Regression (Baseline)** | $50.65 | $79.03 | 0.3797 |
| **Random Forest (Tuned)** | **$43.09** | **$70.29** | **0.5027** |
| **XGBoost (Tuned)** | $43.72 | $70.63 | 0.4931 |
| **MLP V2 (Batch Norm)** | $45.54 | $76.40 | 0.4717 |

#### **Conclusion**
*   **Champion Model:** The **Tuned Random Forest** is our best-performing model, explaining approximately **50.3%** of the price variance with the lowest error rates.
*   **Key Insight:** Tree-based ensemble methods (Random Forest and XGBoost) significantly outperformed the deep learning and linear approaches for this tabular dataset.
*   **Stability:** The high consistency between validation and test scores across all models confirms that our data preprocessing and outlier handling were successful.